In [ ]:
import numpy as np
import pandas as pd
import torch
from torchinfo import summary
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
!pip install torchinfo

## Creating NN  

In [ ]:
# create model class

class Model(nn.Module):

  def __init__(self, num_features):
    super().__init__()

    # ---> nn.Linear(input, output)
    # self.linear1 = nn.Linear(num_features, 3)
    # self.relu = nn.ReLU()
    # self.linear2 = nn.Linear(3, 1)
    # self.sigmoid = nn.Sigmoid()


    self.network = nn.Sequential(
        nn.Linear(num_features, 3),
        nn.ReLU(),
        nn.Linear(3, 1),
        nn.Sigmoid()
    )



  def forward(self, features):

    # out = self.linear1(features)
    # out = self.relu(out)
    # out = self.linear2(out)
    # out = self.sigmoid(out)

    out = self.network(features)

    return out


In [ ]:
# create dataset
features = torch.rand(10, 5)

# create model
model = Model(features.shape[1])

# call model for forward pass
# model.forward(features)
model(features)

tensor([[0.3756],
        [0.3951],
        [0.4150],
        [0.3655],
        [0.3838],
        [0.4157],
        [0.3985],
        [0.3913],
        [0.3951],
        [0.4082]], grad_fn=<SigmoidBackward0>)

In [ ]:
# weights
# print(model.linear1.weight)
# print(model.linear1.bias)

AttributeError: 'Model' object has no attribute 'linear1'

In [ ]:
summary(model)

Layer (type:depth-idx)                   Param #
Model                                    --
├─Sequential: 1-1                        --
│    └─Linear: 2-1                       18
│    └─ReLU: 2-2                         --
│    └─Linear: 2-3                       4
│    └─Sigmoid: 2-4                      --
Total params: 22
Trainable params: 22
Non-trainable params: 0

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv")

In [ ]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [ ]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

# print(x_train.dtypes)

# print('\n\n', y_train.dtypes)

In [ ]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)   # .fit_transform is used on the training data
x_test = scaler.transform(x_test)         # .transform() is used on test or validation data.

In [ ]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:
# Numpy arrays to PyTorch tensors
x_train_tensor = torch.from_numpy(x_train)
x_test_tensor = torch.from_numpy(x_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)



x_train_tensor = x_train_tensor.to(torch.float32)
x_test_tensor = x_test_tensor.to(torch.float32)
y_train_tensor = y_train_tensor.to(torch.float32)
y_test_tensor = y_test_tensor.to(torch.float32)



print(x_train_tensor.dtype, y_train_tensor.dtype)

torch.float32 torch.float32


### Defining the model

In [ ]:
# print(x_train_tensor.shape)

class MySimpleNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()

    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()


  def forward(self, features):
    out = self.linear(features)
    out = self.sigmoid(out)
    return out



## Important Parameters

In [ ]:
learning_rate = 0.1
epochs = 100

In [ ]:
loss_function = nn.BCELoss()

## Training Pipeline

In [ ]:
# create model
model = MySimpleNN(x_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

print(x_train_tensor.dtype, y_train_tensor.dtype)
# define loop
for epoch in range(epochs):

  # forward pass
  y_pred = model(x_train_tensor)
  # print(y_pred.shape)

  # loss calculate
  # print(y_pred.shape, y_train_tensor.shape)
  loss = loss_function(y_pred, y_train_tensor.unsqueeze(-1))


  # clearing the grad for the next run
  # model.linear.weight.grad.zero_()
  # model.linear.bias.grad.zero_()
  optimizer.zero_grad()


  # backward pass
  loss.backward()


  # parameters udpate
  # with torch.no_grad():
  #   model.linear.weight -= learning_rate * model.linear.weight.grad
  #   model.linear.bias -= learning_rate * model.linear.bias.grad
  optimizer.step()



  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, loss: {loss.item()}')

torch.float32 torch.float32
Epoch: 1, loss: 0.46813836693763733
Epoch: 2, loss: 0.3977026641368866
Epoch: 3, loss: 0.3526447117328644
Epoch: 4, loss: 0.3208935856819153
Epoch: 5, loss: 0.29706069827079773
Epoch: 6, loss: 0.2783623933792114
Epoch: 7, loss: 0.2632047235965729
Epoch: 8, loss: 0.2506040930747986
Epoch: 9, loss: 0.23991800844669342
Epoch: 10, loss: 0.2307078242301941
Epoch: 11, loss: 0.22266286611557007
Epoch: 12, loss: 0.21555650234222412
Epoch: 13, loss: 0.20921915769577026
Epoch: 14, loss: 0.20352114737033844
Epoch: 15, loss: 0.19836144149303436
Epoch: 16, loss: 0.19366012513637543
Epoch: 17, loss: 0.18935289978981018
Epoch: 18, loss: 0.1853874921798706
Epoch: 19, loss: 0.18172083795070648
Epoch: 20, loss: 0.17831718921661377
Epoch: 21, loss: 0.1751464456319809
Epoch: 22, loss: 0.17218323051929474
Epoch: 23, loss: 0.1694057732820511
Epoch: 24, loss: 0.16679547727108002
Epoch: 25, loss: 0.16433624923229218
Epoch: 26, loss: 0.16201403737068176
Epoch: 27, loss: 0.1598166227

In [ ]:
model.linear.weight.shape

torch.Size([1, 30])

# Evaluation

In [ ]:
with torch.no_grad():
  y_pred = model(x_test_tensor)
  y_pred = (y_pred > 0.5).float()

  accuracy = (y_pred == y_test_tensor.unsqueeze(-1)).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.9649122953414917
